In [1]:
import numpy as np
import pandas as pd
import rasterio as rio
from matplotlib import pyplot as plt
from tqdm import tqdm
import joblib
from pathlib import Path

In [2]:
ROOT = Path.cwd().parent
DATA = ROOT / "data"
RESULTS = ROOT / "results"
FIGURES = ROOT / "figures"
FEATURE_DIR = RESULTS / "features"

In [3]:
# Load the saved global model and its feature names
final_model = joblib.load(RESULTS / 'trained-models' / 'rf-global.pkl')
feature_names_full = joblib.load(RESULTS / 'trained-models' / 'rf-global-features.pkl')

In [4]:
country_pseudo_values = pd.read_csv(DATA / 'country-pseudo-values.csv')

In [5]:
country_raster_       = rio.open(RESULTS / 'country-rasters' / 'country-raster-eck4-1km.tif')

In [6]:
country_raster        = country_raster_.read(1)

In [7]:
region_mask = rio.open(RESULTS / 'region-masks' / 'region-mask-1km.tif').read(1)

In [8]:
feature_list_1km = dict({
    'elevation': FEATURE_DIR / 'elevation-1km.tif',
    'slope':     FEATURE_DIR / 'slope-1km.tif', 
    'population': RESULTS / 'population-rasters' / 'population-2020-1km.tif',
    'accessibility': FEATURE_DIR / 'accessibility-1km.tif',
    'D2water': FEATURE_DIR / 'distance-to-water-1km.tif', 
    'forest': FEATURE_DIR / 'forest-2020-1km.tif',
    'cropland': FEATURE_DIR / 'cropland-2019-1km.tif', 
    'water': FEATURE_DIR / 'water-mask-1km.tif', 
    'su': FEATURE_DIR / 'su-1km.tif',
    'vb': FEATURE_DIR / 'vb-1km.tif',
    'ss': FEATURE_DIR / 'ss-1km.tif',
    'pb': FEATURE_DIR / 'pb-1km.tif',
    'sm': FEATURE_DIR / 'sm-1km.tif', 
    'sc': FEATURE_DIR / 'sc-1km.tif', 
    'va': FEATURE_DIR / 'va-1km.tif',
    'mt': FEATURE_DIR / 'mt-1km.tif',
    'pa': FEATURE_DIR / 'pa-1km.tif',
    'vi': FEATURE_DIR / 'vi-1km.tif',
    'wb': FEATURE_DIR / 'wb-1km.tif',
    'py': FEATURE_DIR / 'py-1km.tif',
    'pi': FEATURE_DIR / 'pi-1km.tif',
    'ev': FEATURE_DIR / 'ev-1km.tif',
    'nd': FEATURE_DIR / 'nd-1km.tif', 
    'ig': FEATURE_DIR / 'ig-1km.tif', 
    'PA_Ia': RESULTS / 'protected-area-rasters' / 'protected-area-Ia-mask.tif',
    'PA_Ib': RESULTS / 'protected-area-rasters' / 'protected-area-Ib-mask.tif',
    'PA_II': RESULTS / 'protected-area-rasters' / 'protected-area-II-mask.tif',
    'PA_III': RESULTS / 'protected-area-rasters' / 'protected-area-III-mask.tif',
    'PA_IV': RESULTS / 'protected-area-rasters' / 'protected-area-IV-mask.tif',
    'PA_V': RESULTS / 'protected-area-rasters' / 'protected-area-V-mask.tif',
    'PA_VI': RESULTS / 'protected-area-rasters' / 'protected-area-VI-mask.tif',
    'urban': FEATURE_DIR / 'urban-area-2020-1km.tif',
    'bare': FEATURE_DIR / 'bare-land-2020-1km.tif',
    'building_stock': FEATURE_DIR / 'building-mineral-stock-1km.tif',
})

In [9]:
selected_features = ['elevation', 'slope', 'population', 'accessibility', 'D2water', 'forest', 'cropland', 'water', 'su', 'vb', 'ss', 'pb', 'sm', 'sc', 'va', 'mt', 'pa', 'vi', 'wb', 'py', 'pi', 'ev', 'nd', 'ig', 'PA_Ia', 'PA_Ib', 'PA_II', 'PA_III', 'PA_IV', 'PA_V', 'PA_VI', 'urban', 'bare', 'building_stock']
print(len(selected_features)) 

34


In [10]:
region_code = dict({
    'Middle East': 1, 
    'Latin America and the Caribbean': 2,
    'North Africa': 3, 
    'West Africa': 4,
    'East Africa': 5,
    'Central Africa': 6,
    'Southern Africa': 7,
    'Europe': 8, 
    'South-eastern Asia': 9, 
    'Oceania': 10,
    'Southern Asia': 11, 
    'Eastern Asia': 12, 
    'Central and Northern Asia': 13, 
    'Northern America': 14,
    'China': 15
})

In [11]:
# Sanity check — should be 49 (34 env + 15 region one-hot)
print(f'Model expects {len(feature_names_full)} features')
print(f'Region one-hot columns: {[f for f in feature_names_full if f.startswith("reg_")]}')

# Identify region one-hot columns and their order
region_oh_cols = [f for f in feature_names_full if f.startswith('reg_')]
region_ids_in_order = [int(c.split('_')[1]) for c in region_oh_cols]
n_env = len(selected_features)
n_regions = len(region_oh_cols)

Model expects 49 features
Region one-hot columns: ['reg_1', 'reg_2', 'reg_3', 'reg_4', 'reg_5', 'reg_6', 'reg_7', 'reg_8', 'reg_9', 'reg_10', 'reg_11', 'reg_12', 'reg_13', 'reg_14', 'reg_15']


In [12]:
def model_predict_proba_safe(model, X, chunk_size=500_000):
    """Predict in chunks to avoid memory blowup on large countries."""
    n = X.shape[0]
    if n <= chunk_size:
        return model.predict_proba(X)[:, 1]
    out = np.empty(n, dtype=np.float32)
    for start in range(0, n, chunk_size):
        end = min(start + chunk_size, n)
        out[start:end] = model.predict_proba(X[start:end])[:, 1]
    return out

In [13]:
TILE = 2048
BIG_THRESHOLD = 10_000_000  # pixels in bounding box; tune as needed

pred_global = np.zeros_like(region_mask, dtype=np.float32)
water_col = selected_features.index('water')

for i in tqdm(range(1, 256)):

    if i in [161, 213]:
        continue
    matches = country_pseudo_values[country_pseudo_values['value'] == i]
    if len(matches) == 0:
        continue
    
    the_region = matches['region'].values[0]
    if pd.isna(the_region):
        # print(f"Skipping country value={i} (no region)")
        continue

    region_id = region_code[matches['region'].values[0]]
    region_col_idx = region_ids_in_order.index(region_id)

    idx = np.where(country_raster == i)
    if len(idx[0]) == 0:
        continue
    i1, i2 = idx[0].min(), idx[0].max() + 1
    j1, j2 = idx[1].min(), idx[1].max() + 1

    bbox_size = (i2 - i1) * (j2 - j1)

    if bbox_size <= BIG_THRESHOLD:
        # ---- Small country: original whole-window approach ----
        a_window = ((i1, i2), (j1, j2))
        feats = []
        for feature in selected_features:
            with rio.open(feature_list_1km[feature]) as src:
                feats.append(src.read(1, window=a_window).astype(np.float32).ravel())

        X_env = np.stack(feats, axis=1)
        # mask_valid = ~np.isnan(X_env).any(axis=1)
        mask_valid = ~np.isnan(X_env).any(axis=1) & (X_env[:, water_col] == 0)
        X_env_valid = X_env[mask_valid]

        if X_env_valid.shape[0] == 0:
            continue

        region_oh = np.zeros((X_env_valid.shape[0], n_regions), dtype=np.float32)
        region_oh[:, region_col_idx] = 1.0
        X_full = np.hstack([X_env_valid, region_oh])
        p_valid = model_predict_proba_safe(final_model, X_full)

        pred = np.full(X_env.shape[0], np.nan, dtype=np.float32)
        pred[mask_valid] = p_valid
        pred = pred.reshape(i2 - i1, j2 - j1)

        submask = (country_raster[i1:i2, j1:j2] == i)
        pred_global[i1:i2, j1:j2][submask] = pred[submask]

    else:
        # ---- Big country: tile the bounding window ----
        for ti in range(i1, i2, TILE):
            for tj in range(j1, j2, TILE):
                ti2, tj2 = min(ti + TILE, i2), min(tj + TILE, j2)
                h, w = ti2 - ti, tj2 - tj

                submask = (country_raster[ti:ti2, tj:tj2] == i)
                if not submask.any():
                    continue

                feats = []
                for feature in selected_features:
                    with rio.open(feature_list_1km[feature]) as src:
                        feats.append(src.read(1, window=((ti, ti2), (tj, tj2)))
                                     .astype(np.float32).ravel())
                X_env = np.stack(feats, axis=1)

                # mask_valid = ~np.isnan(X_env).any(axis=1) & submask.ravel()
                mask_valid = ~np.isnan(X_env).any(axis=1) & submask.ravel() & (X_env[:, water_col] == 0)

                if not mask_valid.any():
                    continue
                X_env_valid = X_env[mask_valid]

                region_oh = np.zeros((X_env_valid.shape[0], n_regions), dtype=np.float32)
                region_oh[:, region_col_idx] = 1.0
                X_full = np.hstack([X_env_valid, region_oh])
                p_valid = model_predict_proba_safe(final_model, X_full)

                pred = np.full(h * w, np.nan, dtype=np.float32)
                pred[mask_valid] = p_valid
                pred = pred.reshape(h, w)

                pred_global[ti:ti2, tj:tj2][submask] = pred[submask]

100%|██████████| 255/255 [16:50<00:00,  3.96s/it]


In [15]:
pred_scaled = np.nan_to_num(pred_global, nan=0.0)
pred_scaled = np.clip(pred_scaled, 0, 1)
pred_scaled = (pred_scaled * 10000).astype(np.uint16)

In [17]:
with rio.open(
    RESULTS / 'global-mining-suitability-1km.tif',
    'w',
    driver='GTiff',
    height=pred_scaled.shape[0],
    width=pred_scaled.shape[1],
    count=1,
    dtype=pred_scaled.dtype,
    crs=country_raster_.crs,
    transform=country_raster_.transform,
    compress='lzw',
) as dst:
    dst.write(pred_scaled, 1)